### RAG(Retrieval-Augmented Generation), 검색 증강 생성
- 생성형 AI가 학습하지 않은 최신 정보나 특정 조직의 내부 데이터를 바탕으로 답변할 수 있도록 돕는 기술이다.
- R (Retrieval): 질문과 관련된 문서를 방대한 데이터베이스에서 검색한다.
- A (Augmentation): 검색된 정보를 질문과 결합하여 프롬프트를 보강한다.
- G (Generation): 보강된 맥락을 바탕으로 최종 답변을 생성한다.

### RAG 개발 순서
1. 문서 불러오기
2. 문서 분할하기
3. 임베딩
4. Vector DB 생성 및 저장
5. 검색기 생성
6. 프롬프트 생성
7. LLM 생성
8. 체인 실행

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.globals import set_llm_cache
from langchain_community.cache import RedisSemanticCache
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

C:\Users\sokko\anaconda3\envs\llmplz\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [4]:
# RecursiveCharacterTextSplitter: 문서를 작은 조각으로 나눔
from langchain_text_splitters import RecursiveCharacterTextSplitter
# FAISS: 벡터DB임. 문서 조각들을 임베딩(숫자로 변환)해서 저장하고, 질문과 비슷한 문서를 찾는데사용.
from langchain_community.vectorstores import FAISS
# StrOutputParser: LLM답변에서 문자열만 꺼냄
from langchain_core.output_parsers import StrOutputParser
# RunnablePassthrough 사용자 질문을 체인안에서 그대로 넘길때 사용
from langchain_core.runnables import RunnablePassthrough
# PromptTemplate: RAG용 프롬프트 양식 만듦
from langchain_core.prompts import PromptTemplate
# ChatOpenAI: 답변 생성용 LLM
# OpenAIEmbeddings: 문서와 질문을 벡터로 바꾸는 임베딩 모델
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [6]:
# from langchain_community.document_loaders import TextLoader

# # 텍스트 로더 생성
# loader = TextLoader("경로")

# # 문서 로드
# docs = loader.load()

In [ ]:
# from langchain_community.document_loaders.csv_loader import CSVLoader

# # CSV 로더 생성
# loader = CSVLoader(file_path="경로")

# # 데이터 로드
# docs = loader.load()

In [ ]:
# from langchain_community.document_loaders import UnstructuredExcelLoader

# # Excel 로더 생성
# loader = UnstructuredExcelLoader("경로", mode="elements")

# # 문서 로드
# docs = loader.load()

In [5]:
# PyMuPDFLoader: pdf를 읽는 로더
from langchain_community.document_loaders import PyMuPDFLoader

# 1단계, 문서 로드
FILE_PATH = "./documents/ai커리큘럼.pdf"
# 로더생성
loader = PyMuPDFLoader(FILE_PATH)
# pdf를 실제로 읽음
docs = loader.load()

# docs는 결과물임. Document 객체들의 리스트

print(f"문서의 페이지 수: {len(docs)}")

문서의 페이지 수: 3


In [4]:
# Document 객체는 크게 두 부분으로 되어있음

# Document(
#       page_content="문서 내용",
#       metadata={...}
#   )

# page_content는 실제텍스트
# metadata는 부가정보

In [5]:
print(docs)

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-08-07T08:56:01+09:00', 'source': './documents/ai커리큘럼.pdf', 'file_path': './documents/ai커리큘럼.pdf', 'total_pages': 3, 'format': 'PDF 1.5', 'title': '', 'author': 'Admin', 'subject': '', 'keywords': '', 'moddate': '2025-08-07T08:56:01+09:00', 'trapped': '', 'modDate': "D:20250807085601+09'00'", 'creationDate': "D:20250807085601+09'00'", 'page': 0}, page_content='1. 과정 개요 \n\uf0b7 \n대상: 초보자, 비전공자, 고졸자 \n\uf0b7 \n목표: \no PyTorch 기반 머신러닝/딥러닝 실무 능력 배양 \no LangChain + RAG 를 활용한 LLM 문서검색 챗봇 구축 \no SAM2 등 멀티모달 AI 모델 활용 실습 \no MCP(Model Context Protocol) 기반 AI 모델 통합 및 API 설계 \no FastAPI 를 이용한 AI 서비스 상용화 프로젝트 완성 \no 포트폴리오 및 취업 역량 강화 \n \n2. 커리큘럼 요약 \n월 \n주차 \n주요 주제 \n실습 키워드 \n1 월 1~4 주 \nPython & PyTorch 기초 \nPython, Numpy, Pandas, 선형회귀 \n2 월 5~8 주 \nPyTorch 딥러닝 기본 \nMLP, CNN, 분류 모델 구현 \n3 월 9~12 주 LangChain + RAG 활용 \nGPT API, LangChain, FAISS, 문서 \n검색 \n4 월 13~16 주 멀티모달 AI 실습 (SAM2 \n

In [6]:
print(docs[0])

page_content='1. 과정 개요 
 
대상: 초보자, 비전공자, 고졸자 
 
목표: 
o PyTorch 기반 머신러닝/딥러닝 실무 능력 배양 
o LangChain + RAG 를 활용한 LLM 문서검색 챗봇 구축 
o SAM2 등 멀티모달 AI 모델 활용 실습 
o MCP(Model Context Protocol) 기반 AI 모델 통합 및 API 설계 
o FastAPI 를 이용한 AI 서비스 상용화 프로젝트 완성 
o 포트폴리오 및 취업 역량 강화 
 
2. 커리큘럼 요약 
월 
주차 
주요 주제 
실습 키워드 
1 월 1~4 주 
Python & PyTorch 기초 
Python, Numpy, Pandas, 선형회귀 
2 월 5~8 주 
PyTorch 딥러닝 기본 
MLP, CNN, 분류 모델 구현 
3 월 9~12 주 LangChain + RAG 활용 
GPT API, LangChain, FAISS, 문서 
검색 
4 월 13~16 주 멀티모달 AI 실습 (SAM2 
포함) 
CLIP, DINOv2, SAM2, Hugging Face 
5 월 17~20 주 MCP + FastAPI 상용화 
MCP 설계, FastAPI API 개발, Docker 
배포 
 
3. 주차별 상세 커리큘럼 
1~2 개월차: 기초 파운데이션 
 
1 주차: Python 기본 문법, Git, VSCode 활용법 
 
2 주차: 함수, 클래스, 자료구조, 알고리즘 기본 
 
3 주차: Numpy, Pandas, 데이터 시각화 기초 
 
4 주차: PyTorch 텐서 기초, 선형 회귀 모델 구현 
 
5 주차: 로지스틱 회귀, 분류 문제 해결 
 
6 주차: MLP 신경망 구성, 손실 함수 이해 
 
7 주차: CNN 이해 및 이미지 분류 실습 
 
8 주차: 중간 프로젝트 - 이미지 또는 텍스트 분류기 완성 
3 개월차: LLM 과 LangChain + RAG' metadata={'producer': 'Microsoft® Word 2016', 'creato

In [7]:
print(docs[0].page_content)

1. 과정 개요 
 
대상: 초보자, 비전공자, 고졸자 
 
목표: 
o PyTorch 기반 머신러닝/딥러닝 실무 능력 배양 
o LangChain + RAG 를 활용한 LLM 문서검색 챗봇 구축 
o SAM2 등 멀티모달 AI 모델 활용 실습 
o MCP(Model Context Protocol) 기반 AI 모델 통합 및 API 설계 
o FastAPI 를 이용한 AI 서비스 상용화 프로젝트 완성 
o 포트폴리오 및 취업 역량 강화 
 
2. 커리큘럼 요약 
월 
주차 
주요 주제 
실습 키워드 
1 월 1~4 주 
Python & PyTorch 기초 
Python, Numpy, Pandas, 선형회귀 
2 월 5~8 주 
PyTorch 딥러닝 기본 
MLP, CNN, 분류 모델 구현 
3 월 9~12 주 LangChain + RAG 활용 
GPT API, LangChain, FAISS, 문서 
검색 
4 월 13~16 주 멀티모달 AI 실습 (SAM2 
포함) 
CLIP, DINOv2, SAM2, Hugging Face 
5 월 17~20 주 MCP + FastAPI 상용화 
MCP 설계, FastAPI API 개발, Docker 
배포 
 
3. 주차별 상세 커리큘럼 
1~2 개월차: 기초 파운데이션 
 
1 주차: Python 기본 문법, Git, VSCode 활용법 
 
2 주차: 함수, 클래스, 자료구조, 알고리즘 기본 
 
3 주차: Numpy, Pandas, 데이터 시각화 기초 
 
4 주차: PyTorch 텐서 기초, 선형 회귀 모델 구현 
 
5 주차: 로지스틱 회귀, 분류 문제 해결 
 
6 주차: MLP 신경망 구성, 손실 함수 이해 
 
7 주차: CNN 이해 및 이미지 분류 실습 
 
8 주차: 중간 프로젝트 - 이미지 또는 텍스트 분류기 완성 
3 개월차: LLM 과 LangChain + RAG


In [8]:
print(docs[0].metadata)

{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-08-07T08:56:01+09:00', 'source': './documents/ai커리큘럼.pdf', 'file_path': './documents/ai커리큘럼.pdf', 'total_pages': 3, 'format': 'PDF 1.5', 'title': '', 'author': 'Admin', 'subject': '', 'keywords': '', 'moddate': '2025-08-07T08:56:01+09:00', 'trapped': '', 'modDate': "D:20250807085601+09'00'", 'creationDate': "D:20250807085601+09'00'", 'page': 0}


In [9]:
docs[0].__dict__ 
# docs 는 리스트임. docs 리스트 안 첫번째([0]) Document 객체를 내부 dict로 확인.
# 제이슨이나 대괄호로 알고리즘 짤때, 객체 내부구조 확인할때 사용하겠지

{'id': None,
 'metadata': {'producer': 'Microsoft® Word 2016',
  'creator': 'Microsoft® Word 2016',
  'creationdate': '2025-08-07T08:56:01+09:00',
  'source': './documents/ai커리큘럼.pdf',
  'file_path': './documents/ai커리큘럼.pdf',
  'total_pages': 3,
  'format': 'PDF 1.5',
  'title': '',
  'author': 'Admin',
  'subject': '',
  'keywords': '',
  'moddate': '2025-08-07T08:56:01+09:00',
  'trapped': '',
  'modDate': "D:20250807085601+09'00'",
  'creationDate': "D:20250807085601+09'00'",
  'page': 0},
 'page_content': '1. 과정 개요 \n\uf0b7 \n대상: 초보자, 비전공자, 고졸자 \n\uf0b7 \n목표: \no PyTorch 기반 머신러닝/딥러닝 실무 능력 배양 \no LangChain + RAG 를 활용한 LLM 문서검색 챗봇 구축 \no SAM2 등 멀티모달 AI 모델 활용 실습 \no MCP(Model Context Protocol) 기반 AI 모델 통합 및 API 설계 \no FastAPI 를 이용한 AI 서비스 상용화 프로젝트 완성 \no 포트폴리오 및 취업 역량 강화 \n \n2. 커리큘럼 요약 \n월 \n주차 \n주요 주제 \n실습 키워드 \n1 월 1~4 주 \nPython & PyTorch 기초 \nPython, Numpy, Pandas, 선형회귀 \n2 월 5~8 주 \nPyTorch 딥러닝 기본 \nMLP, CNN, 분류 모델 구현 \n3 월 9~12 주 LangChain + RAG 활용 \nGPT API, LangChain, FAISS, 

In [6]:
# 2단계, 문서 분할
# 청크끼리 얼마나겹치게 할래 => 문맥유지를 위해 앞뒤 50자
# 보통 500으로 하고 10~20%로 설정함.

# chunk_size는 문서 조각 하나의 크기임.
# chunk_size=500이면 한조각을 대략 500자 정도로 자름.

# chunk_size
# LLM의 기억력과 검색의 정확도를 결정짓는다.
# 너무 작으면 맥락이 잘려서 LLM의 이해도가 떨어지고
# 너무 크면 불필요한 정보가 많이 섞여서 답변의 초점이 흐려진다.
# 500~800자: 3~4문단

# chunk_overlap은 조각사이 겹치는 부분.
# 문장을 자르다가 중요한 말이 경계에 걸릴수 있는데, 겹침이 없으면
# chunk1: 피자는 느끼해
# chunk2: 그래서 싫어 
# 가 되어 맥락 끊김. 겹치게 자르면
# chunk1: 피자는 느끼해 난 그래서
# chunk2: 느끼해 난 그래서 싫어 
# 공통 부분이 생겨서 문맥 손실 줄어듦

# chunk_overlap
# 단락을 분할하면서 생기는 정보의 손실을 최소화하기 위해 연결고리를 만들어준다.
# 피자는 느끼해 난 그래서 싫어
# chunk1: 피자는 느끼해 난 그래서
# chunk2: 느끼해 난 그래서 싫어
# overlap: 느끼해 난 그래서
# 보통 chunk_size의 10~20%정도가 적당하다고 판단한다.

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"분할된 청크(조각의 수): {len(split_documents)}")

# => pdf 3페이지가 5개의 작은 조각으로 나뉜것이다.

분할된 청크(조각의 수): 5


In [8]:
# 3단계, 임베딩
# 텍스트를 숫자 벡터로 바꾸는작업
embeddings = OpenAIEmbeddings()

In [9]:
# 4단계, 벡터 DB

# split_documents를 전부 임베딩하고, FAISS 벡터 DB에 저장함.

# 즉, 아랙와 같은 작업이 일어남
# 문서 조각 1 -> 임베딩 벡터 -> FAISS 저장
# 문서 조각 2 -> 임베딩 벡터 -> FAISS 저장
# 문서 조각 3 -> 임베딩 벡터 -> FAISS 저장
# ...

vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)
# 그럼 이제 질문을 넣으면 유사도검색이 되겠다.

In [10]:
vectorstore.similarity_search("PyTorch")[0].page_content

# PyTorch 와 의미적 가장 가까운 문서 조각을 찾고, 그 중 첫 번째 결과의 본문만 출력
# similarity_search("PyTorch") 결과는 Document 리스트임. [Document(...), Document(...), ...]
# 그래서 [0] 첫번째 결과를 꺼내고 .page_content 본문만 봄

'1. 과정 개요 \n\uf0b7 \n대상: 초보자, 비전공자, 고졸자 \n\uf0b7 \n목표: \no PyTorch 기반 머신러닝/딥러닝 실무 능력 배양 \no LangChain + RAG 를 활용한 LLM 문서검색 챗봇 구축 \no SAM2 등 멀티모달 AI 모델 활용 실습 \no MCP(Model Context Protocol) 기반 AI 모델 통합 및 API 설계 \no FastAPI 를 이용한 AI 서비스 상용화 프로젝트 완성 \no 포트폴리오 및 취업 역량 강화 \n \n2. 커리큘럼 요약 \n월 \n주차 \n주요 주제 \n실습 키워드 \n1 월 1~4 주 \nPython & PyTorch 기초 \nPython, Numpy, Pandas, 선형회귀 \n2 월 5~8 주 \nPyTorch 딥러닝 기본 \nMLP, CNN, 분류 모델 구현 \n3 월 9~12 주 LangChain + RAG 활용 \nGPT API, LangChain, FAISS, 문서 \n검색 \n4 월 13~16 주 멀티모달 AI 실습 (SAM2 \n포함)'

In [11]:
# 5단계, 검색기 생성

# 1. k=1로 설정: 아주 핵심적인 정보 하나만 딱 본다.

# 장점: 답변이 집중됨.
# 단점: 주변 맥락이 부족할 수 있음

# 2. k=5로 설정: 관련 문서조각 5개 가져옴. 주변 맥락까지 폭넓게 읽어서 풍부하게 답변한다.

# 장점: 정보가 풍부함.
# 단점: 불필요한 문서가 섞일 수 있음.

# retriever_1 = vectorstore.as_retriever(search_kwargs={"k": 1})
# retriever_5 = vectorstore.as_retriever(search_kwargs={"k": 5})

# 3. 기본값: k=4
retriever = vectorstore.as_retriever(search_kwargs={"k":1})
retriever.invoke("프로젝트")

[Document(id='7b7b7abe-f260-4034-98c7-27826ad51cb1', metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-08-07T08:56:01+09:00', 'source': './documents/ai커리큘럼.pdf', 'file_path': './documents/ai커리큘럼.pdf', 'total_pages': 3, 'format': 'PDF 1.5', 'title': '', 'author': 'Admin', 'subject': '', 'keywords': '', 'moddate': '2025-08-07T08:56:01+09:00', 'trapped': '', 'modDate': "D:20250807085601+09'00'", 'creationDate': "D:20250807085601+09'00'", 'page': 2}, page_content='\uf0b7 \n완성된 AI 프로젝트 2~3 건 (GitHub 공개) \n\uf0b7 \nREST API 서버 및 문서 \n\uf0b7 \n포트폴리오 발표 자료 및 데모 영상 \n\uf0b7 \n기술 이력서 및 자기소개서 템플릿 \n\uf0b7 \n모의 면접 및 실무 발표 시뮬레이션 \n \n7. 부록: 취업 전략 및 팁 \n\uf0b7 \nAI 직무 맞춤형 이력서 작성법 \n\uf0b7 \nGitHub 프로젝트 관리법 \n\uf0b7 \n기술 면접 예상 질문 및 답변 가이드')]

In [12]:
# 6단계, 프롬프트 생성
prompt_template = PromptTemplate.from_template(
    """
    귀하는 제공된 참고 문헌을 바탕으로 질문에 답하는 정보 분석 전문가입니다. 
    답변 시 반드시 제시된 문맥(Context) 내의 정보만을 활용하십시오. 
    만약 주어진 자료만으로 답변이 어렵다면, 추측하지 말고 
    '제공된 정보로는 확인이 불가능하다'고 명확히 밝히십시오. 
    모든 응답은 한국어로 작성합니다.

    #Context: 
    {context}
    
    #Question:
    {question}
    
    #Answer:
    """
    # context에는 검색된 문서조각이, question에는 사용자의 질문이 들어감. 
    # => #Context: [검색된 문서내용] #Question: 3개월차에는 무엇을 배우나요?
)

In [13]:
# 7단계, LLM 생성
llm = ChatOpenAI(
    model_name="gpt-5.4-nano",
    temperature=0
)
# 답변 생성용 모델 만들기. 문서기반 답변이니 창의성을 0으로 준다.

In [14]:
# ### 8단계, 체인 생성

#   RAG 체인의 핵심은 사용자의 질문 하나를 받아서 두 가지 값으로 나누어 준비하는 것이다.

#   1. `context`: 질문과 관련된 문서 검색 결과
#   2. `question`: 사용자가 입력한 원래 질문

#   ```python
#   chain = (
#       {"context": retriever, "question": RunnablePassthrough()}
#       | prompt_template
#       | llm
#       | StrOutputParser()
#   )


#  {"context": retriever, "question": RunnablePassthrough()} 의미

#   LCEL에서는 체인 맨앞에 딕셔너리를 둘 수 있다.

#   {"context": retriever, "question": RunnablePassthrough()}

#   이 딕셔너리는 일반적인 고정 dict가 아니라, 입력값을 받아서 새로운 dict를 만드는 처리 단계

#   예를 들어 사용자가 질문으로
#   question = "3개월차에는 무엇을 배우나요?"이 입력값 하나가 두 갈래로 전달된다.

#   1. context: retriever로 검색
#   "context": retriever 이건 사용자 질문을 retriever에 넣음

#   retriever.invoke("3개월차에는 무엇을 배우나요?") 그러면 벡터 DB에서 질문과 관련된 문서 조각을 검색

#   검색 결과는 이런 식
#   [
#       Document(page_content="3개월차: LLM과 LangChain + RAG 활용 ...")
#   ]이 검색 결과가 context 값이 된다.

#   2. question: 원래 질문 그대로 유지

#   "question": RunnablePassthrough()
#   RunnablePassthrough()는 입력값을 아무것도 바꾸지 않고 그대로 통과시킨다.

#   입력이:"3개월차에는 무엇을 배우나요?"라면 출력도 그대로:
#   "3개월차에는 무엇을 배우나요?"이다. 이 값이 question 값

#   결과적으로 만들어지는 dict

#   따라서
#   {"context": retriever, "question": RunnablePassthrough()}
#   는 내부적으로 이런 일을 한다

#   input_question = "3개월차에는 무엇을 배우나요?"
#   {
#       "context": retriever.invoke(input_question),
#       "question": input_question
#   }

#   최종적으로 이런 형태의 dict가 만들어진다.

#   {
#       "context": [
#           Document(page_content="3개월차: LLM과 LangChain + RAG 활용 ...")
#       ],
#       "question": "3개월차에는 무엇을 배우나요?"
#   }

#   그 다음
#   이 dict는 다음 단계인 prompt_template으로 전달된다.

#   | prompt_template

#   프롬프트에는 두 변수가 있다.

#   {context}
#   {question}

#   따라서 방금 만든 dict의 값들이 각각 들어간다.

#   #Context:
#   검색된 문서 내용

#   #Question:
#   3개월차에는 무엇을 배우나요?

#   #Answer:
#   그 다음 LLM이 이 프롬프트를 보고 답변한다.

#   | llm

#   마지막으로 StrOutputParser()가 답변 내용만 문자열로

#   | StrOutputParser()

#  전체 흐름

#   사용자 질문 입력
#       ↓
#   {"context": retriever, "question": RunnablePassthrough()}
#       ↓
#   context에는 관련 문서 검색 결과가 들어감
#   question에는 원래 질문이 그대로 들어감
#       ↓
#   prompt_template이 context와 question을 합쳐 RAG 프롬프트 생성
#       ↓
#   llm이 검색된 문서를 참고하여 답변 생성
#       ↓
#   StrOutputParser가 답변 문자열만 반환

#   요약

#   {"context": retriever, "question": RunnablePassthrough()}
#   는 사용자의 질문 하나를 받아서 아래처럼 바꾸는 단계이다.
#   {
#       "context": "질문과 관련된 문서 검색 결과",
#       "question": "사용자가 입력한 원래 질문"
#   }
#   RAG에서는 이 패턴을 자주 사용한다.

#   chain = (
#       {"context": retriever, "question": RunnablePassthrough()}
#       | prompt_template
#       | llm
#       | StrOutputParser()
#   )

In [14]:
# 8단계, 체인 생성

chain = (
    # 여기 딕셔너리들이 들어가도됨.
    # retriever에도 입력값(질문)이, RunnablePassthrough에도 입력값이 들어감.
    # => 입력질문을 검색기에 넣었으니 retriever.invoke("[입력값]")으로 결과가 문서검색결과로 바뀜.
    # "context": [Document(...), Document(...)] 로, question은 아무 처리 없이 입력질문을 넘김
    # 입력: "3개월차엔 뭘배워?"
    # RunnablePassthrough() 를 거치니
    # 출력: "3개월차엔 뭘배워?"
    # "question": "3개월차엔 뭘배워?"
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

# 그럼이제 캐싱. 질문이 키값. 답변이 밸류값.
# 유사도검색

In [15]:
question = "LangChain 체인 설계는 몇 주차에 배우는가?"
answer = chain.invoke(question)
print(answer)

# 질문 입력
#   -> retriever가 관련 문서 검색
#   -> 검색 결과가 context에 들어감
#   -> 원래 질문이 question에 들어감
#   -> prompt_template 완성
#   -> llm 답변 생성
#   -> 문자열로 출력

제공된 문맥에 따르면 **LangChain 체인 설계 및 기본 체인 구현은 10 주차**에 배우는 것으로 확인됩니다.


In [16]:
from langchain.globals import set_llm_cache
from langchain_community.cache import RedisSemanticCache

set_llm_cache(RedisSemanticCache(
  redis_url="redis://localhost:6380",
  embedding=embeddings,  # 이미 만든 HuggingFaceEmbeddings 재사용
  score_threshold=0.1
))

print("redis semantic cache ready")

redis semantic cache ready


In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# docker run -d --name redis-stack -p 6380:6379 -p 8001:8001 redis/redis-stack-server:latest
        # redis-stack 컨테이너를 백그라운드로 실행
        # 로컬 6380 포트를 컨테이너 내부 6379 포트에 연결
        # 로컬 8001 포트를 RedisInsight UI에 연결
# docker exec -it redis-stack redis-cli
        # 실행중인 Redis 컨테이너 안으로 들어가서 redis-cli 실행하는 명령어

# LangChain의 LLM캐시 설정
from langchain.globals import set_llm_cache
# 질문이 정확하지 않아도 의미가 비슷하면 이전 답변 재사용
from langchain_community.cache import RedisSemanticCache
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

# 1. 캐시용 임베딩 모델
# 질문을 벡터로 바꾸기 위하 모델
cache_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-nli")

# 2. 시맨틱 캐시 설정 (Docker로 띄운 Redis Stack 연결)
# 랭체인 LLM 호출 결과를 RedisSemanticCache 에 저장해라
set_llm_cache(RedisSemanticCache(
    # 로컬 레디스스택에 연결
    redis_url="redis://localhost:6380",
    embedding=cache_embeddings,
    # 얼마나 비슷해야 같은 질문으로 볼지 정하는 기준.
    # score_threshold는 낮을수록 더 엄격함.
    # 낮음: 아주 비슷한 질문만 캐시히트.
    # 높음: 조금 비슷해도 캐시히트.
    score_threshold=0.1  # RAG는 답변의 정확도가 중요하므로 임계값을 보수적으로(낮게) 설정
))

# 단계 1: 문서 로드(Load Documents)
FILE_PATH = "./documents/ai커리큘럼.pdf"
loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()
print(f"문서의 페이지수: {len(docs)}")

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"분할된 청크의수: {len(split_documents)}")

# 단계 3: 임베딩(Embedding) 생성
# 우리 컴퓨터 내에서 모델을 돌리므로 데이터가 외부로 나가지 않음
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli", # 한국어 성능이 검증된 오픈소스 모델
    model_kwargs={'device': 'cpu'}    # GPU가 있다면 'cuda'
)

# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어 생성
# FAISS 벡터DB 생성
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성한다.
# 1. k=1로 설정: 아주 핵심적인 정보 하나만 딱 본다.
# retriever_1 = vectorstore.as_retriever(search_kwargs={"k": 1})
# 2. k=5로 설정: 주변 맥락까지 폭넓게 읽어서 풍부하게 답변한다.
# retriever_5 = vectorstore.as_retriever(search_kwargs={"k": 5})
# 3. 기본값: k=4
retriever = vectorstore.as_retriever()

# 단계 6: 프롬프트 생성(Create Prompt)
prompt = PromptTemplate.from_template(
    """귀하는 제공된 참고 문헌을 바탕으로 질문에 답하는 정보 분석 전문가입니다. 
    답변 시 반드시 제시된 문맥(Context) 내의 정보만을 활용하십시오. 
    만약 주어진 자료만으로 답변이 어렵다면, 추측하지 말고 
    '제공된 정보로는 확인이 불가능하다'고 명확히 밝히십시오. 
    모든 응답은 한국어로 작성합니다.

#Context: 
{context}

#Question:
{question}

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
llm = ChatOpenAI(model_name="gpt-5.4-mini", temperature=0)

# 단계 8: 체인(Chain) 생성
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "LangChain 체인 설계는 몇 주차에 배우는가?"
response = chain.invoke(question)
print(response)

문서의 페이지수: 3
분할된 청크의수: 5
LangChain 체인 설계는 **10주차**에 배웁니다.


In [5]:
question = "LangChain 체인 설계는 몇 주차에 배우는가?"
response = chain.invoke(question)
print(response)

LangChain 체인 설계는 **10주차**에 배웁니다.


In [6]:
question = "랭체인 체인 설계는 몇 주차에 배워?"
response = chain.invoke(question)
print(response)

LangChain 체인 설계는 **10주차**에 배웁니다.


In [7]:
import time

start = time.time()
question = "랭체인 배울 예정인데 체인 설계는 언제배우지?"
response = chain.invoke(question)
print(response)
print(f"소요시간: {time.time() - start:.4f}초")

LangChain 체인 설계는 **10주차**에 배웁니다.
소요시간: 0.8943초
